# Secondary Genetic Code Explorer — Google Colab

This notebook uses the original dissertation Excel dataset to build a reproducible amino-acid identity classifier and a Streamlit scientist-facing app. It explicitly checks duplicates and uses **sequence-grouped validation** to reduce train/test leakage.

In [ ]:
!pip -q install streamlit scikit-learn joblib pandas numpy


## 1. Upload the original Excel dataset
Upload `allalafullseq.xlsx` when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()
xlsx_path = next(iter(uploaded.keys()))
print("Using:", xlsx_path)


## 2. Parse the workbook without assuming column names
The supplied workbook has `SPECIES`, `amino acid`, followed by nucleotide-position columns 1–85.

In [ ]:
import zipfile, xml.etree.ElementTree as ET, re
import pandas as pd

NS='http://schemas.openxmlformats.org/spreadsheetml/2006/main'
def colnum(s):
    x=0
    for ch in s:
        x=x*26+ord(ch)-64
    return x

def parse_sgc_xlsx(path):
    with zipfile.ZipFile(path) as z:
        shared=[]
        if 'xl/sharedStrings.xml' in z.namelist():
            root=ET.fromstring(z.read('xl/sharedStrings.xml'))
            for si in root.findall(f'{{{NS}}}si'):
                shared.append(''.join(t.text or '' for t in si.iter(f'{{{NS}}}t')))
        root=ET.fromstring(z.read('xl/worksheets/sheet1.xml'))
        sheet_data=root.find(f'{{{NS}}}sheetData')
        records=[]
        for r in list(sheet_data)[1:]:
            cells={}
            for c in r.findall(f'{{{NS}}}c'):
                col=re.match(r'[A-Z]+',c.attrib['r']).group()
                typ=c.attrib.get('t'); v=c.find(f'{{{NS}}}v'); val=''
                if typ=='s' and v is not None: val=shared[int(v.text)]
                elif typ=='inlineStr': val=''.join(t.text or '' for t in c.iter(f'{{{NS}}}t'))
                elif v is not None: val=v.text
                cells[col]=val.strip()
            if not cells: continue
            species=cells.get('A','').strip(); aa=cells.get('B','').strip()
            pos=[]
            for i in range(3,88):
                n=i; col=''
                while n:
                    n,rem=divmod(n-1,26); col=chr(65+rem)+col
                b=cells.get(col,'').strip().upper().replace('U','T')
                if b and b not in {'A','C','G','T'}: b='N'
                pos.append(b)
            seq=''.join(b for b in pos if b)
            rec={'species':species,'amino_acid':aa,**{f'pos_{i+1}':b for i,b in enumerate(pos)},'sequence':seq,'length':len(seq)}
            records.append(rec)
    return pd.DataFrame(records)

df=parse_sgc_xlsx(xlsx_path)
print(df.shape)
df.head()


## 3. Audit the data
The duplicate audit is important: identical rows should not be allowed to leak into both training and validation.

In [ ]:
print("Rows:", len(df))
print("Classes:", df['amino_acid'].nunique())
print("Species:", df['species'].value_counts())
print("\nClass counts:\n", df['amino_acid'].value_counts())
print("\nSequence lengths:\n", df['length'].value_counts().sort_index())

work=df[df['length']>=59].copy()
work=work.drop_duplicates(subset=['sequence','amino_acid']).reset_index(drop=True)
conflicts=work.groupby('sequence')['amino_acid'].nunique()
print("\nRows after exact sequence/class dedup:",len(work))
print("Unique sequences:",work['sequence'].nunique())
print("Sequences associated with >1 amino-acid label:",(conflicts>1).sum())


## 4. Train an interpretable baseline
Each nucleotide position is categorical and is one-hot encoded. Sequence length is included as a numeric feature. The split is grouped by exact sequence.

In [ ]:
import numpy as np, joblib, json
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

pos_cols=[f'pos_{i}' for i in range(1,86)]
X=work[pos_cols+['length']].copy()
for c in pos_cols: X[c]=X[c].replace('', 'MISSING')
y=work['amino_acid']; groups=work['sequence']
pre=ColumnTransformer([
    ('seq',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),pos_cols),
    ('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),['length'])
])
model=Pipeline([('preprocess',pre),('model',LogisticRegression(max_iter=700,class_weight='balanced',solver='lbfgs'))])
tr,te=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=42).split(X,y,groups=groups))
model.fit(X.iloc[tr],y.iloc[tr])
pred=model.predict(X.iloc[te])
acc=accuracy_score(y.iloc[te],pred); mf1=f1_score(y.iloc[te],pred,average='macro')
print(f'Grouped holdout accuracy: {acc:.4f}')
print(f'Grouped holdout macro-F1: {mf1:.4f}')
print(classification_report(y.iloc[te],pred,zero_division=0))


## 5. Inspect the confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
fig,ax=plt.subplots(figsize=(12,12))
ConfusionMatrixDisplay.from_predictions(y.iloc[te],pred,xticks_rotation=90,ax=ax,cmap='Blues',colorbar=False)
plt.title('Sequence-grouped holdout confusion matrix')
plt.show()


## 6. Fit the final deployment model and save metadata

In [ ]:
model.fit(X,y)
joblib.dump(model,'sgc_model.joblib')
metadata={
 'n_original_rows':int(len(df)),
 'n_training_rows_after_exact_dedup':int(len(work)),
 'n_unique_sequences':int(work['sequence'].nunique()),
 'n_classes':int(y.nunique()),
 'classes':sorted(y.unique().tolist()),
 'species':sorted(work['species'].unique().tolist()),
 'grouped_holdout_accuracy':float(acc),
 'grouped_holdout_macro_f1':float(mf1),
 'test_rows':int(len(te)),
 'split_note':'20% GroupShuffleSplit grouped by exact sequence.',
 'model':'One-hot nucleotide positions 1-85 + length; class-balanced logistic regression.'
}
with open('sgc_model_metadata.json','w') as f: json.dump(metadata,f,indent=2)
metadata


## 7. Write the Streamlit app

In [ ]:
app_code = 'import io\nimport json\nimport re\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\n\nBASE = Path(__file__).resolve().parent\nMODEL_PATH = BASE / "sgc_model.joblib"\nMETA_PATH = BASE / "sgc_model_metadata.json"\n\nst.set_page_config(page_title="Secondary Genetic Code Explorer", page_icon="🧬", layout="wide")\n\n@st.cache_resource\ndef load_model():\n    return joblib.load(MODEL_PATH)\n\n@st.cache_data\ndef load_metadata():\n    with open(META_PATH, "r", encoding="utf-8") as f:\n        return json.load(f)\n\nmodel = load_model()\nmeta = load_metadata()\nPOS_COLS = [f"pos_{i}" for i in range(1, 86)]\n\n\ndef clean_sequence(seq: str) -> str:\n    seq = re.sub(r"[^A-Za-z]", "", str(seq)).upper().replace("U", "T")\n    return seq\n\n\ndef sequence_to_features(seq: str) -> pd.DataFrame:\n    seq = clean_sequence(seq)\n    vals = [seq[i] if i < len(seq) else "MISSING" for i in range(85)]\n    row = {c: v for c, v in zip(POS_COLS, vals)}\n    row["length"] = len(seq)\n    return pd.DataFrame([row])\n\n\ndef predict_sequence(seq: str):\n    X = sequence_to_features(seq)\n    pred = model.predict(X)[0]\n    probs = model.predict_proba(X)[0]\n    classes = model.classes_\n    order = np.argsort(probs)[::-1]\n    ranked = pd.DataFrame({"Amino acid": classes[order], "Probability": probs[order]})\n    return pred, ranked\n\n\ndef parse_fasta(text: str):\n    records = []\n    name = None\n    chunks = []\n    for line in text.splitlines():\n        line = line.strip()\n        if not line:\n            continue\n        if line.startswith(">"):\n            if name is not None:\n                records.append((name, "".join(chunks)))\n            name = line[1:].strip() or f"sequence_{len(records)+1}"\n            chunks = []\n        else:\n            chunks.append(line)\n    if name is not None:\n        records.append((name, "".join(chunks)))\n    return records\n\n\ndef position_summary(seq: str):\n    seq = clean_sequence(seq)\n    def base(pos):\n        return seq[pos-1] if len(seq) >= pos else "—"\n    anticodon = seq[33:36] if len(seq) >= 36 else "—"\n    return {\n        "Length": len(seq),\n        "Position 3": base(3),\n        "Anticodon region (34–36)": anticodon,\n        "Position 70": base(70),\n        "Position 72": base(72),\n        "Position 73": base(73),\n        "G3:T70 motif": "Present" if len(seq) >= 70 and base(3)=="G" and base(70)=="T" else "Not present",\n    }\n\nst.title("🧬 Secondary Genetic Code Explorer")\nst.caption("A scientist-facing prototype derived from a Ph.D. tRNA sequence dataset: 20 amino-acid classes, nucleotide positions 1–85, and six species.")\nst.info("Research prototype: predictions are generated by a class-balanced logistic-regression baseline trained on the supplied dissertation dataset. They are for exploration, not experimental or clinical decision-making.")\n\nm1,m2,m3,m4=st.columns(4)\nm1.metric("Original rows", f"{meta[\'n_original_rows\']:,}")\nm2.metric("Unique training rows", f"{meta[\'n_training_rows_after_exact_dedup\']:,}")\nm3.metric("Grouped holdout accuracy", f"{meta[\'grouped_holdout_accuracy\']:.1%}")\nm4.metric("Macro-F1", f"{meta[\'grouped_holdout_macro_f1\']:.1%}")\n\ntab1, tab2, tab3, tab4 = st.tabs(["Single sequence", "Batch analysis", "Model insights", "About / limitations"])\n\nwith tab1:\n    st.subheader("Analyze one tRNA sequence")\n    default = "GCGTTGGTGGTATAGTGGTGAGCATAGCTGCCTAAGCAGTTGACCCGGGTTCGATTCCCGGCCAACGCA"\n    seq = st.text_area("Paste DNA or RNA sequence", value=default, height=120, help="U is converted to T. Non-letter characters are removed.")\n    cleaned = clean_sequence(seq)\n    if cleaned:\n        invalid = sorted(set(cleaned) - set("ACGTN"))\n        if invalid:\n            st.error(f"Unsupported characters after cleaning: {invalid}")\n        elif len(cleaned) > 85:\n            st.warning("The training table contains positions 1–85. Only the first 85 positions are represented by the model, while length is retained as a feature.")\n        if st.button("Predict amino-acid identity", type="primary"):\n            pred, ranked = predict_sequence(cleaned)\n            left,right = st.columns([1,2])\n            with left:\n                st.metric("Predicted class", pred)\n                top_prob=float(ranked.iloc[0]["Probability"])\n                st.metric("Top-class probability", f"{top_prob:.1%}")\n                st.write("**Sequence features**")\n                st.dataframe(pd.DataFrame(position_summary(cleaned).items(), columns=["Feature","Value"]), hide_index=True, use_container_width=True)\n            with right:\n                st.write("**Top predictions**")\n                top = ranked.head(8).copy()\n                top["Probability"] = top["Probability"].map(lambda x: f"{x:.2%}")\n                st.dataframe(top, hide_index=True, use_container_width=True)\n                st.bar_chart(ranked.head(10).set_index("Amino acid")["Probability"])\n\nwith tab2:\n    st.subheader("Batch prediction")\n    st.write("Upload a FASTA file, or a CSV/TSV containing a column named `sequence`. Optional identifier columns are preserved.")\n    up = st.file_uploader("Upload batch sequences", type=["fasta","fa","fas","csv","tsv","txt"])\n    if up:\n        name=up.name.lower()\n        if name.endswith((".fasta",".fa",".fas",".txt")):\n            text=up.getvalue().decode("utf-8", errors="replace")\n            records=parse_fasta(text)\n            batch=pd.DataFrame(records, columns=["id","sequence"])\n        else:\n            sep="\\t" if name.endswith(".tsv") else ","\n            batch=pd.read_csv(up, sep=sep)\n            if "sequence" not in batch.columns:\n                st.error("The table must contain a `sequence` column.")\n                st.stop()\n        if len(batch):\n            results=[]\n            for _,row in batch.iterrows():\n                seqv=clean_sequence(row["sequence"])\n                if not seqv:\n                    pred=""; conf=np.nan\n                else:\n                    p, ranked=predict_sequence(seqv); pred=p; conf=float(ranked.iloc[0]["Probability"])\n                rec=row.to_dict(); rec.update({"clean_sequence":seqv,"length":len(seqv),"predicted_amino_acid":pred,"top_probability":conf})\n                results.append(rec)\n            result=pd.DataFrame(results)\n            st.success(f"Processed {len(result):,} sequences")\n            st.dataframe(result.head(200), use_container_width=True)\n            st.download_button("Download predictions CSV", result.to_csv(index=False).encode("utf-8"), "sgc_predictions.csv", "text/csv")\n\nwith tab3:\n    st.subheader("Model design and validation")\n    st.write("**Baseline:** one-hot encoded nucleotide identities at positions 1–85 plus sequence length, followed by class-balanced logistic regression.")\n    st.write("**Leakage control:** exact duplicate sequence/class rows were removed before training, and evaluation used a 20% GroupShuffleSplit grouped by exact sequence. Identical sequences therefore cannot occur in both training and test sets.")\n    st.write(f"**Grouped holdout:** {meta[\'grouped_holdout_accuracy\']:.3f} accuracy; {meta[\'grouped_holdout_macro_f1\']:.3f} macro-F1 on {meta[\'test_rows\']:,} held-out rows.")\n    st.write("**Classes:** " + ", ".join(meta["classes"]))\n    st.write("**Species:** " + ", ".join(meta["species"]))\n    st.code("sequence → standardized positions → one-hot encoding → logistic regression → class probabilities → scientist-facing output", language="text")\n    st.warning("A stronger biological generalization test would hold out entire species or evolutionary groups. The included Colab notebook shows how to extend validation rather than treating this baseline as final biological evidence.")\n\nwith tab4:\n    st.subheader("What this app demonstrates")\n    st.markdown("""\n- Translation of a scientific sequence-analysis workflow into a reusable user-facing model.\n- Standardized inputs, deterministic preprocessing, documented assumptions, reproducible prediction, and downloadable outputs.\n- Explicit separation of training/validation from deployment.\n- A simple, interpretable baseline before adding more complex models.\n""")\n    st.subheader("Limitations")\n    st.markdown("""\n- Positions are treated as sequential columns from the supplied dataset; the app does not infer canonical tRNA structural numbering or perform sequence alignment.\n- The data contain repeated sequences and uneven class/species distributions; grouped validation reduces duplicate leakage but does not eliminate all sources of dataset bias.\n- Prediction confidence is model probability, not biological certainty.\n- This prototype should not be presented as a validated production or clinical model.\n""")\n'
with open('app.py','w',encoding='utf-8') as f:
    f.write(app_code)
print('Wrote app.py')

In [ ]:
with open('requirements.txt','w') as f:
    f.write('streamlit>=1.38\npandas>=2.0\nnumpy>=1.24\nscikit-learn>=1.4\njoblib>=1.3\n')
print(open('requirements.txt').read())

## 8. Launch Streamlit in Colab
Colab does not expose port 8501 directly. The following uses LocalTunnel. Run the first cell, then the second. If LocalTunnel asks for a tunnel password, open the printed `loca.lt/mytunnelpassword` URL in another tab to see the current Colab public IP.

In [ ]:
!npm -q install -g localtunnel
!streamlit run app.py --server.port 8501 --server.headless true > /tmp/streamlit.log 2>&1 &
print("Streamlit started on port 8501")


In [ ]:
!npx localtunnel --port 8501


## 9. Download deployment files
These are the files you need for a Streamlit Community Cloud/GitHub deployment.

In [ ]:
import shutil
shutil.make_archive('secondary_genetic_code_explorer','zip','.')
from google.colab import files
files.download('secondary_genetic_code_explorer.zip')
